# Machine Body Exclusion & Image Preprocessing

This notebook demonstrates image preprocessing on fluidized bed pellet images:
- **Background (No Pellets)**: `Real-Data/exp 1000 bak light only no pellets/`
- **Foreground (With Pellets)**: `Real-Data/Exp1000 only external light/`

### Pipeline Steps:
1. **Baseline Loading**: Read Arabic / non-ASCII paths cleanly.
2. **Method 1: Background Subtraction (`cv2.absdiff`)**: Cancels static machine housing.
3. **Method 2: Static Machine Masking**: Excludes physical boundaries/walls.
4. **Method 3: Blur & Distortion Removal**: Filters out out-of-focus, motion-blurred, and elongated/distorted pellets using **Laplacian variance (sharpness)**, **edge gradient energy**, and **circularity / aspect-ratio metrics**.

In [ ]:
import os
import glob
import cv2
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

def load_image(filepath):
    """Load grayscale image safely handling Unicode/Arabic filenames on Windows."""
    img_bytes = np.fromfile(filepath, dtype=np.uint8)
    return cv2.imdecode(img_bytes, cv2.IMREAD_GRAYSCALE)

# Paths to datasets
no_pellets_dir = "Real-Data/exp 1000 bak light only no pellets"
pellets_dir = "Real-Data/Exp1000 only external light"

no_pellets_files = sorted(glob.glob(os.path.join(no_pellets_dir, "*.bmp")))
pellets_files = sorted(glob.glob(os.path.join(pellets_dir, "*.bmp")))

print(f"Found {len(no_pellets_files)} background images (no pellets).")
print(f"Found {len(pellets_files)} foreground images (with pellets).")

## 1. Load Baseline (Machine Background) & Current Frame (Machine + Pellets)

In [ ]:
# Load empty machine baseline and frame with pellets
bg_img = load_image(no_pellets_files[0])
pellet_img = load_image(pellets_files[0])

# Display original images
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(bg_img, cmap='gray')
axes[0].set_title('Empty Machine (No Pellets)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(pellet_img, cmap='gray')
axes[1].set_title('Machine with Pellets (External Light)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 2. Method 1: Background Subtraction (`cv2.absdiff`)

Subtracts the empty machine reference from the pellet frame. Static machine geometry is canceled out, leaving predominantly the pellets.

In [ ]:
# Compute absolute difference
subtracted = cv2.absdiff(pellet_img, bg_img)

# Threshold to suppress illumination noise while preserving pellet pixel intensities
_, subtracted_clean = cv2.threshold(subtracted, 25, 255, cv2.THRESH_TOZERO)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(subtracted, cmap='gray')
axes[0].set_title('Subtracted Image (pellet_img - bg_img)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(subtracted_clean, cmap='gray')
axes[1].set_title('Subtracted + Noise Suppressed (Thresholded)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 3. Method 2: Machine Region Masking

Builds a mask of the machine boundary from the empty reference frame to explicitly mask out external borders and housing.

In [ ]:
# Create binary mask for machine body from baseline image
_, machine_mask = cv2.threshold(bg_img, 40, 255, cv2.THRESH_BINARY_INV)

# Morphological opening to clean up small artifacts
kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
clean_mask = cv2.morphologyEx(machine_mask, cv2.MORPH_OPEN, kernel)

# Apply mask to the subtracted pellet image
masked_result = cv2.bitwise_and(subtracted_clean, subtracted_clean, mask=clean_mask)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(machine_mask, cmap='gray')
axes[0].set_title('1. Static Machine Mask', fontsize=12)
axes[0].axis('off')

axes[1].imshow(subtracted_clean, cmap='gray')
axes[1].set_title('2. Subtracted Pellets', fontsize=12)
axes[1].axis('off')

axes[2].imshow(masked_result, cmap='gray')
axes[2].set_title('3. Final Result (Machine Excluded)', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

## 4. Method 3: Remove Blurred & Distorted Pellets (Blur / Out-of-Focus Filter)

In imaging systems (like fluid beds), fast-moving or out-of-focus particles appear as faint, low-contrast, or smeared/elongated shapes.

We remove them using two complementary techniques:
1. **Pixel-Level Sharpness / Edge Gradient Filter**: Retains sharp, focused edges via Sobel/Laplacian gradient energy.
2. **Particle-Level Shape & Focus Classification**: Analyzes each contour's **Circularity / Aspect Ratio** (distortion) and **Laplacian Variance** (blur) to filter out smeared, non-spherical, or out-of-focus particles.

In [ ]:
# --- 4A. Pixel-Level Sharp Edge Filtering (Sobel Gradient) ---
# Compute gradients in X and Y to measure edge sharpness
grad_x = cv2.Sobel(masked_result, cv2.CV_64F, 1, 0, ksize=3)
grad_y = cv2.Sobel(masked_result, cv2.CV_64F, 0, 1, ksize=3)
gradient_magnitude = cv2.magnitude(grad_x, grad_y)
sharpness_map = np.uint8(np.clip(gradient_magnitude, 0, 255))

# Only retain regions above sharpness threshold
SHARPNESS_THRESH = 30
_, sharp_mask = cv2.threshold(sharpness_map, SHARPNESS_THRESH, 255, cv2.THRESH_BINARY)
sharp_pellets = cv2.bitwise_and(masked_result, masked_result, mask=sharp_mask)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].imshow(masked_result, cmap='gray')
axes[0].set_title('All Particles (Before Blur Filter)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(sharpness_map, cmap='hot')
axes[1].set_title('Edge Sharpness Map (Sobel Energy)', fontsize=12)
axes[1].axis('off')

axes[2].imshow(sharp_pellets, cmap='gray')
axes[2].set_title('Sharp Pellets Only (Blurred Filtered)', fontsize=12)
axes[2].axis('off')

plt.tight_layout()
plt.show()

### 4B. Particle-by-Particle Quality Filtering (Circularity & Focus Variance)

Filters each detected particle individually based on:
- **Min / Max Area**: Excludes tiny background noise specs or huge artifacts.
- **Circularity**: $\text{Circularity} = 4\pi \cdot \frac{\text{Area}}{\text{Perimeter}^2}$ (Removes motion-streaked / distorted pellets).
- **Laplacian Focus Measure**: $\sigma^2(\nabla^2 I)$ (Removes out-of-focus / blurry pellets).

In [ ]:
# Find binary contours from the sharp pellet mask
_, binary_particles = cv2.threshold(masked_result, 30, 255, cv2.THRESH_BINARY)
contours, _ = cv2.findContours(binary_particles, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Create clean output image
clean_particles_img = np.zeros_like(masked_result)
annotated_rgb = cv2.cvtColor(pellet_img, cv2.COLOR_GRAY2RGB)

# Criteria thresholds (tune as needed)
MIN_AREA = 30           # Minimum particle area in pixels
MIN_CIRCULARITY = 0.50  # 1.0 is a perfect circle; distorted / streaked pellets have low circularity
MIN_LAPLACIAN_VAR = 20  # Sharpness / focus threshold (higher = sharper)

valid_count = 0
rejected_count = 0

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < MIN_AREA:
        continue
    
    perimeter = cv2.arcLength(cnt, True)
    if perimeter == 0:
        continue
    
    # 1. Circularity test (distortion check)
    circularity = 4 * np.pi * (area / (perimeter * perimeter))
    
    # 2. Local focus check (Laplacian variance)
    x, y, w, h = cv2.boundingRect(cnt)
    roi = pellet_img[y:y+h, x:x+w]
    laplacian_var = cv2.Laplacian(roi, cv2.CV_64F).var() if roi.size > 0 else 0
    
    # Filter condition
    if circularity >= MIN_CIRCULARITY and laplacian_var >= MIN_LAPLACIAN_VAR:
        # Valid, in-focus, well-formed pellet (Draw Green)
        cv2.drawContours(clean_particles_img, [cnt], -1, 255, thickness=cv2.FILLED)
        cv2.drawContours(annotated_rgb, [cnt], -1, (0, 255, 0), 2)
        valid_count += 1
    else:
        # Blurred / distorted / streaked pellet (Draw Red)
        cv2.drawContours(annotated_rgb, [cnt], -1, (255, 0, 0), 2)
        rejected_count += 1

print(f"Kept (In-Focus / Undistorted): {valid_count} pellets (GREEN)")
print(f"Removed (Blurred / Distorted): {rejected_count} pellets (RED)")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(annotated_rgb)
axes[0].set_title(f'Classification (Green=Sharp Pellets, Red=Blurred/Distorted)', fontsize=12)
axes[0].axis('off')

axes[1].imshow(clean_particles_img, cmap='gray')
axes[1].set_title('Final Clean Binary Mask (No Machine, No Blur)', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.show()